In [ ]:
from pathlib import Path

import polars as pl

pl.Config.set_tbl_hide_column_data_types(True)

In [ ]:
# Common types to reuse
INT_8 = pl.Int8      # Ratings (0-100), Draft Round (0-7), Flags (0/1), Games (0-17)
INT_16 = pl.Int16    # Years (2020-2144), Weight (0-400), Career Games (0-300), Pick # (0-256)
INT_32 = pl.Int32    # Player_ID (0-110k), Salary/Bonus (if in thousands, up to $2.1B)
CAT = pl.Categorical # Positions, Teams, Colleges, Status
SCHEMAS = {
    "player_information": {
        "Player_ID": INT_32,
        "Position": CAT,
        "Draft_Year": INT_16,
        "Draft_Round": INT_8,
        "Drafted_Position": INT_16,
        "Drafted_By": INT_8,  # Team ID
        "Championship_Rings": INT_8,
        "Hall_of_Fame_Flag": INT_8,
        "Career_Games_Played": INT_16,
        "Number_of_Seasons": INT_8,
    },
    "rookies": {
        "Player_ID": INT_32,
        "Position_Group": CAT,
        "College": CAT,
        "Height": INT_8,
        "Weight": INT_16,
        "Dash": INT_16,          # Usually 400-600 (e.g. 4.5s = 450)
        "Solecismic": INT_8,
        "Strength": INT_8,
        "Agility": INT_16,
        "Jump": INT_16,
        "Position_Specific": INT_8,
        "Developed": INT_8,
        "Grade": INT_8,
    },
    "player_record": {
        "Player_ID": INT_32,
        "Team": INT_8,
        "Status": CAT,
        "How_Acquired": CAT,
        "Experience": INT_8,
        "Salary_Year_1": INT_32, # Using 32 to cover up to $2B in thousands
        "Bonus_Year_1": INT_32,
        "Season_Statistics_-_S_Games_Played": INT_8,
        "S_Games_Started": INT_8,
    },
    "players_personal": {
        "Player_ID": INT_32,
        "Current_Overall": INT_8,
        "Future_Overall": INT_8,
        # Note: Skill columns (e.g., Short_Passes) also fit in INT_8
    }
}

In [ ]:
def build_all_panels_optimized(
    data_dir: Path, draft_years: range, max_career_years: int = 25
) -> pl.DataFrame:
    # 1. Build a master lookup of Player_ID -> Draft_Year
    # We find the latest year to get the most complete information
    years = [int(p.name) for p in data_dir.iterdir() if p.is_dir() and p.name.isdigit()]
    latest_available_year = max(years)
            
    info_path = data_dir / str(latest_available_year) / "player_information.csv"

    if info_path.exists():
        print("Building rookie_map with dynamic position lookup...")
        
        # Get the base ID -> Year mapping
        base_info = (
            pl.read_csv(
                info_path, 
                columns=["Player_ID", "Draft_Year"],
                schema_overrides=SCHEMAS["player_information"]
            )
            .filter(pl.col("Draft_Year").is_in(draft_years))
            .unique(subset=["Player_ID"])
        )

        # Now, group by Draft_Year and grab the Position_Group from the specific year folder
        pos_frames = []
        for d_year in draft_years:
            pos_path = data_dir / str(d_year) / "rookies.csv"
            if pos_path.exists():
                df_pos = pl.read_csv(
                    pos_path,
                    columns=["Player_ID", "Position_Group"],
                    schema_overrides=SCHEMAS.get("rookies")
                )
                pos_frames.append(df_pos)
        
        # Combine all position lookups and join to base_info
        if pos_frames:
            all_positions = pl.concat(pos_frames).unique(subset=["Player_ID"])
            rookie_map = base_info.join(all_positions, on="Player_ID", how="left")
        else:
            rookie_map = base_info.with_columns(pl.lit(None).alias("Position_Group"))    
    else:
        return pl.DataFrame()
    # 2. Process each snapshot year directory
    all_snapshot_years = range(min(draft_years), latest_available_year + 1)
    yearly_panels = []
    for snapshot_yr in all_snapshot_years:
        snapshot_dir = data_dir / str(snapshot_yr)
        record_path = snapshot_dir / "player_record.csv"
        ratings_path = snapshot_dir / "players_personal.csv"
        if not record_path.exists():
            continue
        # Load records with optimized schema
        record_lazy = pl.scan_csv(
            record_path, 
            schema_overrides=SCHEMAS["player_record"]
        ).select([
            "Player_ID", "Team", "Salary_Year_1", "Bonus_Year_1", 
            "Season_Statistics_-_S_Games_Played", "S_Games_Started"
        ])
        # Load ratings with optimized schema
        if ratings_path.exists():
            ratings_lazy = pl.scan_csv(
                ratings_path, 
                schema_overrides=SCHEMAS["players_personal"]
            ).select(["Player_ID", "Current_Overall", "Future_Overall"])
            
            df_snapshot = record_lazy.join(ratings_lazy, on="Player_ID", how="left")
        else:
            df_snapshot = record_lazy
        # Process snapshot
        df_snapshot = (
            df_snapshot.join(rookie_map.lazy(), on="Player_ID", how="inner")
            .with_columns([
                pl.lit(snapshot_yr, dtype=pl.Int16).alias("Snapshot_Year"),
                (pl.lit(snapshot_yr, dtype=pl.Int16) - pl.col("Draft_Year") + 1).cast(pl.Int8).alias("Career_Year"),
            ])
            .filter((pl.col("Career_Year") >= 1) & (pl.col("Career_Year") <= max_career_years))
        )
        yearly_panels.append(df_snapshot.collect())
    return pl.concat(yearly_panels) if yearly_panels else pl.DataFrame()

In [ ]:
# Execution for DRAFT003
data_root = Path("../fof8-gen/data/raw/DRAFT003")
final_df = build_all_panels_optimized(data_root, range(2021, 2145))

# Verification
if not final_df.is_empty():
    print(f"Success! Found {final_df.height} rows.")
    print(final_df.filter(pl.col("Season_Statistics_-_S_Games_Played") >= 1).head())


In [ ]:
final_df.schema

In [ ]:
print(final_df.shape[0])
final_df.select(pl.col("Player_ID")).unique()

In [ ]:
query = (
    final_df.lazy()
    .filter(pl.col("Season_Statistics_-_S_Games_Played").sum().over("Player_ID") > 16)
    .group_by("Player_ID")
    .agg(
        pl.first("Position_Group"),
        pl.first("Draft_Year"),
        pl.col("Salary_Year_1").sum().alias("Career_Salary"),
        pl.col("Bonus_Year_1").sum().alias("Career_Bonus"),
        (pl.col("Salary_Year_1") + pl.col("Bonus_Year_1")).sum().alias("Total_Career_Earnings")
    )
    .sort("Total_Career_Earnings", descending=True)
)

results = query.collect()
results

In [ ]:
formatted_results = results.with_columns(
    pl.format(
        "${}", 
        pl.col("Total_Career_Earnings").map_elements(lambda x: f"{x*1000:,.0f}", return_dtype=pl.String)
    ).alias("Total_Earnings_Display")
)
formatted_results

In [ ]:
avg_by_position_df = formatted_results.filter(pl.col("Total_Career_Earnings")>=0).group_by(pl.col("Position_Group")).agg(
    pl.col("Total_Career_Earnings").mean().alias("Average_Career_Earnings")
    )

In [ ]:
with pl.Config(tbl_rows=-1):
    display(avg_by_position_df.sort(pl.col("Average_Career_Earnings"), descending=True))

In [ ]:
import hvplot.polars  # noqa: F401

formatted_results.filter(pl.col("Position_Group") == "QB").select(pl.col("Total_Career_Earnings")).hvplot.hist(y="Total_Career_Earnings", bins="auto")

In [ ]:
formatted_results.filter(pl.col("Total_Career_Earnings")==0)